In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "state-spaces/mamba-2.8b-hf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
)
model.to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/642 [00:00<?, ?it/s]

MambaForCausalLM(
  (backbone): MambaModel(
    (embeddings): Embedding(50280, 2560)
    (layers): ModuleList(
      (0-63): 64 x MambaBlock(
        (norm): MambaRMSNorm(2560, eps=1e-05)
        (mixer): MambaMixer(
          (conv1d): Conv1d(5120, 5120, kernel_size=(4,), stride=(1,), padding=(3,), groups=5120)
          (act): SiLUActivation()
          (in_proj): Linear(in_features=2560, out_features=10240, bias=False)
          (x_proj): Linear(in_features=5120, out_features=192, bias=False)
          (dt_proj): Linear(in_features=160, out_features=5120, bias=True)
          (out_proj): Linear(in_features=5120, out_features=2560, bias=False)
        )
      )
    )
    (norm_f): MambaRMSNorm(2560, eps=1e-05)
  )
  (lm_head): Linear(in_features=2560, out_features=50280, bias=False)
)

In [7]:
prompt = "Explain photosynthesis in one paragraph."
encoded = tokenizer([prompt], return_tensors="pt")
input_ids = encoded["input_ids"].to(DEVICE)

with torch.inference_mode():
    output = model(input_ids=input_ids, use_cache=True, return_dict=True)

cache_obj = getattr(output, "cache_params", None)
if cache_obj is None:
    cache_obj = getattr(output, "past_key_values", None)

type(cache_obj)


transformers.cache_utils.DynamicCache

In [8]:
attention_mask = encoded["attention_mask"].to(DEVICE)

with torch.inference_mode():
    generated_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=100,
        do_sample=False,  # greedy decoding
    )

response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(response)

Explain photosynthesis in one paragraph.

A:

I would say that photosynthesis is the process by which plants use sunlight to convert carbon dioxide and water into sugar.

A:

Photosynthesis is the process by which plants use sunlight to convert carbon dioxide and water into sugar.

A:

Photosynthesis is the process by which plants use sunlight to convert carbon dioxide and water into sugar.




In [9]:
from state_spectrum_sweep.run_experiment_mamba import extract_cache_tensors

views = extract_cache_tensors(cache_obj, subset="all")
print(len(views))
print(views[0].path.name, views[0].tensor.shape, views[0].tensor.dtype, views[0].tensor.device)


128
layers.0.conv_states torch.Size([1, 5120, 4]) torch.bfloat16 cuda:0


In [10]:
ssm_states = extract_cache_tensors(cache_obj, subset="ssm")
conv_states = extract_cache_tensors(cache_obj, subset="conv")

In [11]:
state_tensors = {
    view.path.name: view.tensor.detach().float().cpu().clone()
    for view in views
}


In [12]:
x = state_tensors["layers.23.recurrent_states"]
x.shape, x.layout, x

(torch.Size([1, 5120, 16]),
 torch.strided,
 tensor([[[-1.1902e-03, -1.1368e-03,  1.8311e-04,  ..., -4.0054e-05,
           -3.8719e-04, -1.1292e-03],
          [-1.2302e-04, -8.3542e-04,  4.7922e-05,  ...,  6.7902e-04,
            4.2725e-04, -1.8539e-03],
          [ 1.1215e-03,  4.9744e-03, -6.8283e-04,  ..., -5.0735e-04,
            2.3346e-03,  7.1106e-03],
          ...,
          [-1.5640e-03, -1.2054e-03,  1.7624e-03,  ...,  3.8338e-04,
           -8.2397e-04, -4.8828e-03],
          [ 3.2806e-03, -5.4321e-03, -1.9073e-04,  ..., -6.0654e-04,
            3.3722e-03,  4.3945e-03],
          [ 1.3123e-02,  2.1973e-03, -1.0452e-03,  ..., -1.4019e-04,
            4.0894e-03,  9.0332e-03]]]))

In [13]:
def sparsity_stats(x, tol=1e-6):
    x = x.detach().float().cpu()
    return {
        "shape": tuple(x.shape),
        "zero_frac": float((x == 0).float().mean()),
        "near_zero_frac": float((x.abs() < tol).float().mean()),
        "mean_abs": float(x.abs().mean()),
        "max_abs": float(x.abs().max()),
    }

for name, tensor in list(state_tensors.items())[:10]:
    print(name, sparsity_stats(tensor))


layers.0.conv_states {'shape': (1, 5120, 4), 'zero_frac': 0.0, 'near_zero_frac': 0.0, 'mean_abs': 0.9625492095947266, 'max_abs': 20.0}
layers.0.recurrent_states {'shape': (1, 5120, 16), 'zero_frac': 0.0, 'near_zero_frac': 0.01708984375, 'mean_abs': 0.002323687309399247, 'max_abs': 0.203125}
layers.1.conv_states {'shape': (1, 5120, 4), 'zero_frac': 0.0, 'near_zero_frac': 0.0, 'mean_abs': 0.41360512375831604, 'max_abs': 9.1875}
layers.1.recurrent_states {'shape': (1, 5120, 16), 'zero_frac': 0.0, 'near_zero_frac': 0.003955078311264515, 'mean_abs': 0.005743796471506357, 'max_abs': 0.341796875}
layers.2.conv_states {'shape': (1, 5120, 4), 'zero_frac': 0.0, 'near_zero_frac': 0.0, 'mean_abs': 0.6457487344741821, 'max_abs': 14.4375}
layers.2.recurrent_states {'shape': (1, 5120, 16), 'zero_frac': 0.0, 'near_zero_frac': 0.0024536133278161287, 'mean_abs': 0.001814367831684649, 'max_abs': 0.14453125}
layers.3.conv_states {'shape': (1, 5120, 4), 'zero_frac': 0.0, 'near_zero_frac': 0.0, 'mean_abs': 